In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob

import pandas as pd
import numpy as np

from PIL import Image
import matplotlib.pyplot as plt

import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import Subset
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn

from sklearn.model_selection import train_test_split

from tqdm import tqdm
import random

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform

        self.image_paths = glob.glob(f"{root_dir}/dataset/images/*.jpg")  # Find all images


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        mask_path = self.image_paths[idx].replace('images', 'masks').replace('jpg', 'png')
        image_path = self.image_paths[idx]

        image = Image.open(image_path).convert("RGB")

        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Replace mask values with remapped values
        mask = remap_mask(mask)

        return image, mask       # we return image and mask

In [ ]:
image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor()
])

data = CustomDataset(path, transform = image_transforms, target_transform=mask_transforms)

In [ ]:
train, test = train_test_split(data, test_size=0.2)

In [ ]:
train_loader = DataLoader(train, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Display some
data_iter = iter(train_loader)
images, masks = next(data_iter)


# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i in range (5):
    img = images[i]
    img = img.permute(1, 2, 0) # Convert (C, H, W) to (H, W, C)

    mask = masks[i]
    mask = mask.permute(1, 2, 0)

    axes[0][i].imshow(img)
    axes[1][i].imshow(mask, cmap='gray')

plt.show()

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # we have 8 classes
).to(device)



In [ ]:
print(model)

In [ ]:
# TO DO
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.float)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).to(torch.float)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO